In [1]:
!pip install -q snorkel datasets spacy tqdm wikipedia-api anthropic sec-edgar-downloader nltk
!python -m spacy download en_core_web_sm -q
print("\nInstalare completa. Daca esti in Colab, mergi la Runtime > Restart Runtime, apoi continua de la celula urmatoare.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 45.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

Instalare completa. Daca esti in Colab, mergi la Runtime > Restart Runtime, apoi continua de la celula urmatoare.


In [2]:
import re
import json
import time
import random
import requests
import warnings
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

# ─── Labeluri NER ────────────────────────────────────────────────────────────
# Acestea sunt toate labelurile pe care modelul le va recunoaste.
# Poti adauga labeluri noi fara sa re-antrenezi (avantajul GLiNER).
NER_LABELS = [
    "POLITICIAN",          # Joe Biden, Jerome Powell
    "POLITICAL_PARTY",     # Republican Party, Labour Party
    "POLITICAL_ORG",       # NATO, European Commission, UN
    "FINANCIAL_ORG",       # Federal Reserve, IMF, Goldman Sachs
    "ECONOMIC_INDICATOR",  # GDP, CPI, inflation rate
    "POLICY",              # Quantitative Easing, Green New Deal
    "LEGISLATION",         # Dodd-Frank Act, Inflation Reduction Act
    "MARKET_EVENT",        # 2008 financial crisis, Black Monday
    "CURRENCY",            # USD, EUR, Bitcoin
    "TRADE_AGREEMENT",     # NAFTA, TPP, USMCA
    "GPE",                 # United States, Eurozone, EU
]

# ─── Parametri colectare corpus ──────────────────────────────────────────────
# Modifica aceste valori daca vrei mai multe/putine date
MAX_CC_NEWS_ARTICLES   = 5000   # articole din CC-News
MAX_WIKI_ARTICLES      = 500    # articole Wikipedia
MAX_SEC_FILINGS        = 200    # filings SEC EDGAR
MAX_SENTENCES_PER_DOC  = 20     # propozitii maxime per document
MIN_SENTENCE_LEN       = 30     # caractere minime per propozitie
MAX_SENTENCE_LEN       = 300    # caractere maxime per propozitie

# ─── Directoare output ───────────────────────────────────────────────────────
Path("dataset/raw").mkdir(parents=True, exist_ok=True)
Path("dataset/annotated").mkdir(parents=True, exist_ok=True)
Path("dataset/splits").mkdir(parents=True, exist_ok=True)

print("Configurare completa.")
print(f"Labeluri definite: {NER_LABELS}")

Configurare completa.
Labeluri definite: ['POLITICIAN', 'POLITICAL_PARTY', 'POLITICAL_ORG', 'FINANCIAL_ORG', 'ECONOMIC_INDICATOR', 'POLICY', 'LEGISLATION', 'MARKET_EVENT', 'CURRENCY', 'TRADE_AGREEMENT', 'GPE']


In [3]:
# ─── Sursa 1: CC-News (HuggingFace) ──────────────────────────────────────────
# CC-News contine milioane de articole de stiri din Common Crawl.
# Filtram dupa keywords relevante domeniului nostru.

from datasets import load_dataset

KEYWORDS = [
    # Economic
    "GDP", "inflation", "interest rate", "Federal Reserve", "IMF",
    "World Bank", "stock market", "bond yield", "fiscal policy",
    "monetary policy", "recession", "unemployment", "CPI", "trade deficit",
    # Political
    "president", "senator", "prime minister", "parliament", "election",
    "congress", "legislation", "sanctions", "NATO", "treaty", "minister",
    "European Commission", "White House", "G7", "G20",
]

print("Se incarca CC-News (streaming)...")
cc_news_dataset = load_dataset("cc_news", split="train", streaming=True, trust_remote_code=True)
# streaming = True (datele sunt descarcate on-the-fly -> metoda devine asincrona)
# trust_remote_code = True (acorda permisiuni scriptului python din biblioteca sa descarce, dezarhiveze si formateze articolele de pe serverle sursa,
# folosind un script python intern)

cc_news_texts = []
pbar = tqdm(total=MAX_CC_NEWS_ARTICLES, desc="CC-News") # biblioteca ce arata progresul (bara de progres) a articolelor adaugate in dateset in timp real.

for article in cc_news_dataset:
    text = article.get("text", "").strip()  # strip = elimina spatiile de inceput si final din text; get("text") -> extrage din dictionarul rezultat din CC-News doar
                                            # campul text cu continutul articolului
    if len(text) < 200:
        continue
    if any(kw.lower() in text.lower() for kw in KEYWORDS):  # daca apar orice indicator predefinit in text
        cc_news_texts.append(text)      # adauga textul in dataset
        pbar.update(1)                  # actualizeaza progress barul pt datasetul final
        if len(cc_news_texts) >= MAX_CC_NEWS_ARTICLES:
            break

pbar.close()
print(f"Colectate {len(cc_news_texts)} articole CC-News.")

with open("dataset/raw/cc_news.json", "w") as f:
    json.dump(cc_news_texts, f)
print("Salvat: dataset/raw/cc_news.json")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cc_news' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cc_news' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Se incarca CC-News (streaming)...


CC-News:   0%|          | 0/5000 [00:00<?, ?it/s]

Colectate 5000 articole CC-News.
Salvat: dataset/raw/cc_news.json


In [4]:
# ─── Sursa 2: Wikipedia ───────────────────────────────────────────────────────
# Colectam articole Wikipedia pe teme politice si economice.
# Wikipedia are entitati bine definite si consistente — ideal pentru NER.

import wikipediaapi

# Titluri de articole reprezentative pentru domeniul nostru
WIKI_TOPICS = [
    # Politicieni
    "Joe Biden", "Donald Trump", "Jerome Powell", "Christine Lagarde",
    "Janet Yellen", "Emmanuel Macron", "Olaf Scholz", "Rishi Sunak",
    "Angela Merkel", "Barack Obama", "Xi Jinping", "Vladimir Putin",
    # Organizatii politice
    "NATO", "European Union", "United Nations", "European Commission",
    "Republican Party (United States)", "Democratic Party (United States)",
    "G7", "G20", "BRICS",
    # Organizatii economice
    "International Monetary Fund", "World Bank", "Federal Reserve",
    "European Central Bank", "Bank of England", "Goldman Sachs",
    "JPMorgan Chase", "BlackRock",
    # Concepte economice
    "Quantitative easing", "Inflation", "Gross domestic product",
    "Monetary policy", "Fiscal policy", "Interest rate",
    "Stock market crash", "2008 financial crisis", "Stagflation",
    # Legislatie / Tratate
    "Dodd–Frank Wall Street Reform and Consumer Protection Act",
    "Paris Agreement", "North American Free Trade Agreement",
    "Maastricht Treaty", "Basel III",
]

wiki = wikipediaapi.Wikipedia(language="en", user_agent="NER-Dataset-Builder/1.0")
wiki_texts = []

for title in tqdm(WIKI_TOPICS[:MAX_WIKI_ARTICLES], desc="Wikipedia"):
    try:
        page = wiki.page(title)
        if page.exists() and len(page.text) > 500:
            wiki_texts.append({"title": title, "text": page.text[:5000]})  # primii 5000 chars
        time.sleep(0.2)  # politicos fata de serverele Wikipedia
    except Exception as e:
        print(f"Eroare la '{title}': {e}")

print(f"Colectate {len(wiki_texts)} articole Wikipedia.")
with open("dataset/raw/wikipedia.json", "w") as f:
    json.dump(wiki_texts, f)
print("Salvat: dataset/raw/wikipedia.json")

Wikipedia:   0%|          | 0/43 [00:00<?, ?it/s]

Colectate 43 articole Wikipedia.
Salvat: dataset/raw/wikipedia.json


In [6]:
print(len(WIKI_TOPICS))

43


In [7]:
# ─── Sursa 3: SEC EDGAR ───────────────────────────────────────────────────────
# SEC EDGAR contine rapoartele financiare ale companiilor listate la bursa.
# Acestea contin entitati economice dense: indicatori, organizatii, politici.
#
# NOTA: Daca sec-edgar-downloader ridica probleme in Colab,
# comenteaza aceasta celula — ai suficiente date din CC-News si Wikipedia.

try:
    from sec_edgar_downloader import Downloader
    import os

    # Cateva companii mari cu rapoarte publice relevante
    TICKERS = ["AAPL", "JPM", "GS", "BAC", "MS", "WFC", "C", "BLK"]

    dl = Downloader("NER Research", "ner@research.com", "./dataset/raw/sec_raw")
    sec_texts = []

    for ticker in tqdm(TICKERS[:5], desc="SEC EDGAR"):  # limitam la 5 pentru viteza
        try:
            dl.get("10-K", ticker, limit=1)  # annual report
        except Exception as e:
            print(f"Skip {ticker}: {e}")

    # Citim fisierele descarcate si extragem text
    sec_dir = Path("./dataset/raw/sec_raw")
    if sec_dir.exists():
        for txt_file in list(sec_dir.rglob("*.txt"))[:MAX_SEC_FILINGS]:
            try:
                text = txt_file.read_text(errors="ignore")[:3000]
                if len(text) > 200:
                    sec_texts.append(text)
            except:
                pass

    print(f"Colectate {len(sec_texts)} fragmente SEC EDGAR.")
    with open("dataset/raw/sec_edgar.json", "w") as f:
        json.dump(sec_texts, f)
    print("Salvat: dataset/raw/sec_edgar.json")

except Exception as e:
    print(f"SEC EDGAR nu e disponibil: {e}")
    print("Continuam fara SEC EDGAR — avem suficiente date din celelalte surse.")
    sec_texts = []

SEC EDGAR:   0%|          | 0/5 [00:00<?, ?it/s]

Colectate 5 fragmente SEC EDGAR.
Salvat: dataset/raw/sec_edgar.json


In [8]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])  # doar sentence segmentation

def extract_sentences(texts, source_name, max_per_doc=MAX_SENTENCES_PER_DOC):
    """Extrage propozitii din lista de texte. Filtreaza dupa lungime."""
    sentences = []
    for text in tqdm(texts, desc=f"Segmentare {source_name}", leave=False):
        if isinstance(text, dict):
            text = text.get("text", "")
        try:
            doc = nlp(text[:10000])  # limita procesare per doc
            sents = [
                sent.text.strip()
                for sent in doc.sents
                if MIN_SENTENCE_LEN <= len(sent.text.strip()) <= MAX_SENTENCE_LEN
            ]
            sentences.extend(sents[:max_per_doc])
        except Exception:
            pass
    return sentences

# Incarcam datele raw (in caz ca ai restartat runtime-ul)
with open("dataset/raw/cc_news.json") as f:
    cc_news_texts = json.load(f)
with open("dataset/raw/wikipedia.json") as f:
    wiki_texts = json.load(f)

try:
    with open("dataset/raw/sec_edgar.json") as f:
        sec_texts = json.load(f)
except FileNotFoundError:
    sec_texts = []

# Segmentam fiecare sursa
sents_cc   = extract_sentences(cc_news_texts, "CC-News")
sents_wiki = extract_sentences(wiki_texts, "Wikipedia")
sents_sec  = extract_sentences(sec_texts, "SEC EDGAR")

# Combinam si deduplicam
all_sentences = list(set(sents_cc + sents_wiki + sents_sec))
random.shuffle(all_sentences)

# Cream DataFrame
df_sentences = pd.DataFrame({"text": all_sentences})
df_sentences = df_sentences[df_sentences["text"].str.len().between(MIN_SENTENCE_LEN, MAX_SENTENCE_LEN)]
df_sentences = df_sentences.drop_duplicates(subset="text").reset_index(drop=True)

print(f"Total propozitii dupa segmentare si deduplicare: {len(df_sentences)}")
print(f"  - CC-News:   {len(sents_cc)} propozitii")
print(f"  - Wikipedia: {len(sents_wiki)} propozitii")
print(f"  - SEC EDGAR: {len(sents_sec)} propozitii")
df_sentences.head(5)

Segmentare CC-News:   0%|          | 0/5000 [00:00<?, ?it/s]

Segmentare Wikipedia:   0%|          | 0/43 [00:00<?, ?it/s]

Segmentare SEC EDGAR:   0%|          | 0/5 [00:00<?, ?it/s]

Total propozitii dupa segmentare si deduplicare: 62663
  - CC-News:   73574 propozitii
  - Wikipedia: 853 propozitii
  - SEC EDGAR: 14 propozitii


,text
0,The key question for Eagle and Ahmed: if they ...
1,"Similarly, arrests of non-criminals were also ..."
2,He railed against immigrants in his campaign a...
3,Interested in Trump Administration?
4,"The result, which he eventually termed the ""Ak..."


In [10]:
df_sentences.size

62663

In [11]:
# ─── Gazetteers (liste de entitati cunoscute) ─────────────────────────────────
# Aceste liste servesc drept baza de cunostinte pentru LF-uri.
# Cu cat sunt mai complete, cu atat LF-urile sunt mai precise.

POLITICIANS_GAZETTEER = [
    "Joe Biden", "Donald Trump", "Barack Obama", "George W. Bush",
    "Bill Clinton", "Jerome Powell", "Janet Yellen", "Christine Lagarde",
    "Emmanuel Macron", "Olaf Scholz", "Rishi Sunak", "Angela Merkel",
    "Boris Johnson", "Xi Jinping", "Vladimir Putin", "Narendra Modi",
    "Ursula von der Leyen", "Mario Draghi", "Ben Bernanke", "Alan Greenspan",
    "Larry Summers", "Paul Volcker", "Mark Carney", "Lagarde",
    "Yellen", "Powell", "Bernanke",  # si forme scurte
]

FINANCIAL_ORGS_GAZETTEER = [
    "Federal Reserve", "Fed", "IMF", "International Monetary Fund",
    "World Bank", "European Central Bank", "ECB", "Bank of England",
    "Goldman Sachs", "JPMorgan", "JPMorgan Chase", "Morgan Stanley",
    "BlackRock", "Citigroup", "Bank of America", "Wells Fargo",
    "Deutsche Bank", "Barclays", "HSBC", "Credit Suisse",
    "Bank for International Settlements", "BIS", "FDIC",
    "Securities and Exchange Commission", "SEC",
]

POLITICAL_ORGS_GAZETTEER = [
    "NATO", "United Nations", "UN", "European Union", "EU",
    "European Commission", "European Parliament", "G7", "G20", "BRICS",
    "OECD", "WTO", "World Trade Organization", "Republican Party",
    "Democratic Party", "Labour Party", "Conservative Party",
    "Congress", "Senate", "House of Representatives", "Parliament",
    "White House", "Kremlin", "Bundestag",
]

ECONOMIC_INDICATORS_GAZETTEER = [
    "GDP", "GNP", "CPI", "PPI", "PCE", "PMI",
    "inflation rate", "inflation", "deflation", "stagflation",
    "unemployment rate", "unemployment", "jobless rate",
    "interest rate", "federal funds rate", "discount rate",
    "bond yield", "Treasury yield", "yield curve",
    "trade deficit", "trade surplus", "current account",
    "budget deficit", "national debt", "debt-to-GDP",
]

LEGISLATION_GAZETTEER = [
    "Dodd-Frank", "Dodd–Frank", "Glass-Steagall", "Basel III", "Basel IV",
    "Inflation Reduction Act", "CARES Act", "American Rescue Plan",
    "Sarbanes-Oxley", "Volcker Rule", "GDPR", "MiFID",
    "Patriot Act", "Affordable Care Act", "Stimulus Package",
]

CURRENCIES_GAZETTEER = [
    "USD", "EUR", "GBP", "JPY", "CNY", "CHF", "CAD", "AUD",
    "Bitcoin", "BTC", "Ethereum", "ETH", "dollar", "euro",
    "pound sterling", "yen", "yuan", "renminbi",
]

TRADE_AGREEMENTS_GAZETTEER = [
    "NAFTA", "USMCA", "TPP", "CPTPP", "RCEP",
    "Paris Agreement", "Maastricht Treaty", "Kyoto Protocol",
    "WTO Agreement", "GATT", "TTIP", "CETA",
]

MARKET_EVENTS_GAZETTEER = [
    "2008 financial crisis", "Great Recession", "dot-com bubble",
    "Black Monday", "Black Tuesday", "Great Depression",
    "European debt crisis", "Asian financial crisis",
    "Brexit", "COVID-19 recession", "subprime mortgage crisis",
]

print("Gazetteers incarcate.")
print(f"  Politicieni: {len(POLITICIANS_GAZETTEER)}")
print(f"  Org. financiare: {len(FINANCIAL_ORGS_GAZETTEER)}")
print(f"  Org. politice: {len(POLITICAL_ORGS_GAZETTEER)}")
print(f"  Indicatori econ.: {len(ECONOMIC_INDICATORS_GAZETTEER)}")

Gazetteers incarcate.
  Politicieni: 27
  Org. financiare: 25
  Org. politice: 24
  Indicatori econ.: 25


In [12]:
# ─── Interogare Wikidata pentru politicieni suplimentari ─────────────────────
# Wikidata ne da o lista mult mai extinsa de politicieni decat gazetteer-ul manual.
# NOTA: Aceasta celula face o cerere HTTP catre Wikidata — poate dura 10-30s.

def get_wikidata_politicians(limit=3000):
    """Interogheaza Wikidata pentru lista de politicieni in engleza."""
    query = f"""
    SELECT DISTINCT ?personLabel WHERE {{
      ?person wdt:P31 wd:Q5 .
      ?person wdt:P106 wd:Q82955 .
      ?person wikibase:sitelinks ?links .
      FILTER(?links > 5)
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en" . }}
    }} LIMIT {limit}
    """
    headers = {
        "User-Agent": "NER-Dataset-Builder/1.0 (research project)",
        "Accept": "application/json"
    }
    try:
        r = requests.get(
            "https://query.wikidata.org/sparql",
            params={"query": query, "format": "json"},
            headers=headers,
            timeout=30
        )
        if r.status_code == 200:
            names = [
                item["personLabel"]["value"]
                for item in r.json()["results"]["bindings"]
                if len(item["personLabel"]["value"].split()) >= 2  # doar nume compuse
            ]
            return names
    except Exception as e:
        print(f"Wikidata query esuat: {e}")
    return []

print("Se interogheaza Wikidata...")
wikidata_politicians = get_wikidata_politicians(limit=3000)
# Combinam cu gazetteer-ul manual
ALL_POLITICIANS = list(set(POLITICIANS_GAZETTEER + wikidata_politicians))
# Sortam dupa lungime descrescatoare pentru matching corect (evita match partial)
ALL_POLITICIANS.sort(key=len, reverse=True)

print(f"Total politicieni in baza de cunostinte: {len(ALL_POLITICIANS)}")

Se interogheaza Wikidata...
Total politicieni in baza de cunostinte: 2915


In [13]:
# ─── Labeling Functions Snorkel ───────────────────────────────────────────────
# Fiecare LF returneaza un label (0-9) sau ABSTAIN (-1).
# ABSTAIN inseamna ca LF-ul nu e sigur — Snorkel ignora acest vot.
# Label Model combina voturile tuturor LF-urilor.

from snorkel.labeling import labeling_function
from snorkel.labeling import PandasLFApplier
from snorkel.labeling.model import LabelModel

ABSTAIN          = -1
L_POLITICIAN     = 0
L_POLITICAL_PARTY= 1
L_POLITICAL_ORG  = 2
L_FINANCIAL_ORG  = 3
L_ECONOMIC_IND   = 4
L_POLICY         = 5
L_LEGISLATION    = 6
L_MARKET_EVENT   = 7
L_CURRENCY       = 8
L_TRADE_AGREEMENT= 9

CARDINALITY = 10  # numarul de labeluri (fara ABSTAIN)

# --- LF 1: Politicieni din gazetteer + Wikidata ---
@labeling_function()
def lf_politician_gazetteer(x):
    for name in ALL_POLITICIANS:
        if name.lower() in x.text.lower():
            return L_POLITICIAN
    return ABSTAIN

# --- LF 2: Heuristica titlu + nume pentru politicieni ---
@labeling_function()
def lf_politician_title(x):
    pattern = r"\b(President|Vice President|Senator|Prime Minister|Chancellor|" \
              r"Secretary of State|Foreign Minister|Finance Minister|Governor|" \
              r"Congressman|Congresswoman|Representative)\s+[A-Z][a-z]+"
    return L_POLITICIAN if re.search(pattern, x.text) else ABSTAIN

# --- LF 3: Partide politice ---
@labeling_function()
def lf_political_party(x):
    parties = ["Republican Party", "Democratic Party", "Labour Party",
               "Conservative Party", "Liberal Party", "Green Party",
               "Socialist Party", "Democrats", "Republicans"]
    return L_POLITICAL_PARTY if any(p.lower() in x.text.lower() for p in parties) else ABSTAIN

# --- LF 4: Organizatii politice din gazetteer ---
@labeling_function()
def lf_political_org_gazetteer(x):
    return L_POLITICAL_ORG if any(o.lower() in x.text.lower() for o in POLITICAL_ORGS_GAZETTEER) else ABSTAIN

# --- LF 5: Organizatii financiare din gazetteer ---
@labeling_function()
def lf_financial_org_gazetteer(x):
    return L_FINANCIAL_ORG if any(o.lower() in x.text.lower() for o in FINANCIAL_ORGS_GAZETTEER) else ABSTAIN

# --- LF 6: Regex pentru organizatii financiare ---
@labeling_function()
def lf_financial_org_regex(x):
    pattern = r"\b(central bank|investment bank|hedge fund|private equity|asset manager)s?\b"
    return L_FINANCIAL_ORG if re.search(pattern, x.text, re.IGNORECASE) else ABSTAIN

# --- LF 7: Indicatori economici din gazetteer ---
@labeling_function()
def lf_economic_indicator_gazetteer(x):
    return L_ECONOMIC_IND if any(i.lower() in x.text.lower() for i in ECONOMIC_INDICATORS_GAZETTEER) else ABSTAIN

# --- LF 8: Regex pentru indicatori economici cu valori numerice ---
@labeling_function()
def lf_economic_indicator_numeric(x):
    # Pattern: "GDP grew by 3.2%", "inflation hit 8%", "rate of 5.25%"
    pattern = r"\b(GDP|inflation|unemployment|growth|rate|yield)\b.{0,30}\d+\.?\d*\s*%"
    return L_ECONOMIC_IND if re.search(pattern, x.text, re.IGNORECASE) else ABSTAIN

# --- LF 9: Politici monetare/fiscale ---
@labeling_function()
def lf_policy(x):
    policies = ["quantitative easing", "quantitative tightening", "green new deal",
                "austerity", "stimulus", "bailout", "tapering", "rate hike", "rate cut",
                "fiscal policy", "monetary policy", "economic policy",
                "budget", "tax reform", "deficit spending"]
    return L_POLICY if any(p.lower() in x.text.lower() for p in policies) else ABSTAIN

# --- LF 10: Legislatie ---
@labeling_function()
def lf_legislation(x):
    # Pattern general: "X Act", "X Law", "X Bill"
    act_pattern = r"\b[A-Z][a-zA-Z\s-]+\b(Act|Law|Bill|Regulation|Directive|Amendment)\b"
    if re.search(act_pattern, x.text):
        return L_LEGISLATION
    return L_LEGISLATION if any(l.lower() in x.text.lower() for l in LEGISLATION_GAZETTEER) else ABSTAIN

# --- LF 11: Evenimente de piata ---
@labeling_function()
def lf_market_event(x):
    return L_MARKET_EVENT if any(e.lower() in x.text.lower() for e in MARKET_EVENTS_GAZETTEER) else ABSTAIN

# --- LF 12: Regex pentru crize financiare ---
@labeling_function()
def lf_market_event_regex(x):
    pattern = r"\b(financial crisis|market crash|stock market|bear market|bull market|recession|depression)\b"
    return L_MARKET_EVENT if re.search(pattern, x.text, re.IGNORECASE) else ABSTAIN

# --- LF 13: Valute ---
@labeling_function()
def lf_currency(x):
    if any(c in x.text for c in CURRENCIES_GAZETTEER):
        return L_CURRENCY
    # Pattern: "$1.2 trillion", "€500 billion", "¥100"
    pattern = r"[\$€£¥]\s*\d+"
    return L_CURRENCY if re.search(pattern, x.text) else ABSTAIN

# --- LF 14: Acorduri comerciale ---
@labeling_function()
def lf_trade_agreement(x):
    return L_TRADE_AGREEMENT if any(t.lower() in x.text.lower() for t in TRADE_AGREEMENTS_GAZETTEER) else ABSTAIN

ALL_LFS = [
    lf_politician_gazetteer, lf_politician_title, lf_political_party,
    lf_political_org_gazetteer, lf_financial_org_gazetteer, lf_financial_org_regex,
    lf_economic_indicator_gazetteer, lf_economic_indicator_numeric,
    lf_policy, lf_legislation, lf_market_event, lf_market_event_regex,
    lf_currency, lf_trade_agreement
]

print(f"Definite {len(ALL_LFS)} Labeling Functions.")

Definite 14 Labeling Functions.


In [14]:
# ─── Aplicare LF-uri si Label Model ──────────────────────────────────────────
# PandasLFApplier aplica toate LF-urile pe fiecare rand din DataFrame.
# Rezultatul L_matrix are shape (n_sentences, n_lfs) cu voturi per LF.
# LabelModel combina aceste voturi si produce pseudo-label-uri finale.

print("Se aplica Labeling Functions...")
applier = PandasLFApplier(lfs=ALL_LFS)
L_matrix = applier.apply(df=df_sentences)

print(f"Shape L_matrix: {L_matrix.shape}")
print(f"  Rows: {L_matrix.shape[0]} propozitii")
print(f"  Cols: {L_matrix.shape[1]} LF-uri")

# Statistici LF-uri — coverage si conflicts
from snorkel.labeling import LFAnalysis
lf_analysis = LFAnalysis(L=L_matrix, lfs=ALL_LFS).lf_summary()
print("\nAnaliza LF-uri:")
print(lf_analysis.to_string())

Se aplica Labeling Functions...


100%|██████████| 62663/62663 [11:34<00:00, 90.24it/s]


Shape L_matrix: (62663, 14)
  Rows: 62663 propozitii
  Cols: 14 LF-uri

Analiza LF-uri:
                                  j Polarity  Coverage  Overlaps  Conflicts
lf_politician_gazetteer           0      [0]  0.039066  0.032443   0.023235
lf_politician_title               1      [0]  0.045338  0.036609   0.027401
lf_political_party                2      [1]  0.014347  0.009431   0.009431
lf_political_org_gazetteer        3      [2]  0.366867  0.088729   0.088729
lf_financial_org_gazetteer        4      [3]  0.075723  0.041444   0.041364
lf_financial_org_regex            5      [3]  0.001899  0.001692   0.001612
lf_economic_indicator_gazetteer   6      [4]  0.016565  0.010150   0.010102
lf_economic_indicator_numeric     7      [4]  0.000255  0.000223   0.000176
lf_policy                         8      [5]  0.007117  0.005075   0.005075
lf_legislation                    9      [6]  0.007006  0.004788   0.004788
lf_market_event                  10      [7]  0.001724  0.001452   0.001325


In [15]:
# ─── Antrenare Label Model ────────────────────────────────────────────────────
# LabelModel invata ponderi pentru fiecare LF pe baza corelatiilor dintre ele.
# LF-urile mai precise primesc ponderi mai mari.

print("Se antreneaza Label Model...")
label_model = LabelModel(cardinality=CARDINALITY, verbose=True) # nu e un model de regresie
            # model generativ din Snorkel ce observa matricea de voturi L_matrix si invata din felul in care s-au aplicat
            # Label Functions asupra textului.
            # Dacă LF1 și LF2 dau adesea aceeași etichetă, modelul presupune că ambele sunt corecte.
            # Dacă LF3 dă o etichetă, dar LF1 și LF2 zic că se abțin, modelul învață să aibă încredere în LF3 pentru acel caz specific.
            # Dacă LF4 greșește frecvent (intră în conflict cu celelalte), modelul îi va scădea ponderea (importanța).

# modelul se uita doar la matricea de voturi pt a estima cat de buna este fiecare regula.
label_model.fit(
    L_train=L_matrix,
    n_epochs=500,
    lr=0.001,
    log_freq=100,
    seed=42
)

# Predictii cu probabilitati
pseudo_labels      = label_model.predict(L=L_matrix)
pseudo_probs       = label_model.predict_proba(L=L_matrix)

# aici se adauga la dataframe 2 coloane: snorkel_label adica eticheta catre care textul apartine de entitatea X.
# snorkel_confidence adica probabilitatea de apartenenta la entitatea X.
df_sentences["snorkel_label"]      = pseudo_labels
df_sentences["snorkel_confidence"] = pseudo_probs.max(axis=1)

# Filtram: pastram doar propozitiile cu label non-ABSTAIN si confidence > 0.7
# la noi ABSTAIN ar putea fi considerata o clasa NAN
df_weak = df_sentences[
    (df_sentences["snorkel_label"] != -1) &
    (df_sentences["snorkel_confidence"] >= 0.7)
].copy()

# Mapam label index → label string ca sa avem etichete String (sa putem vedea si in dataframe).
IDX_TO_LABEL = {v: k for k, v in {
    "POLITICIAN": 0, "POLITICAL_PARTY": 1, "POLITICAL_ORG": 2,
    "FINANCIAL_ORG": 3, "ECONOMIC_INDICATOR": 4, "POLICY": 5,
    "LEGISLATION": 6, "MARKET_EVENT": 7, "CURRENCY": 8, "TRADE_AGREEMENT": 9
}.items()}

df_weak["label"] = df_weak["snorkel_label"].map(IDX_TO_LABEL)

print(f"\nPropozitii cu pseudo-label dupa filtrare: {len(df_weak)}")
print("\nDistributie labeluri:")
print(df_weak["label"].value_counts().to_string())

Se antreneaza Label Model...


100%|██████████| 500/500 [00:00<00:00, 780.47epoch/s]



Propozitii cu pseudo-label dupa filtrare: 1261

Distributie labeluri:
label
POLITICIAN            1206
FINANCIAL_ORG           28
MARKET_EVENT            20
ECONOMIC_INDICATOR       7


In [19]:
print(df_weak.size)
df_weak

5044


,text,snorkel_label,snorkel_confidence,label
35,President Donald Trump has been pushing hard f...,0,0.783743,POLITICIAN
137,Days after President Donald Trump deemed Jeff ...,0,0.783743,POLITICIAN
171,Merkel's governments managed the 2008 financia...,7,0.706009,MARKET_EVENT
229,President Donald Trump made a surprise visit t...,0,0.724375,POLITICIAN
262,"On Friday, October 6, Governor John Bel Edward...",0,0.783743,POLITICIAN
...,...,...,...,...
62501,South Korean President Moon Jae-in is meeting ...,0,0.724375,POLITICIAN
62532,Donald Trump’s latest method of deflection is ...,0,0.783743,POLITICIAN
62556,WASHINGTON (AP) — President Donald Trump will ...,0,0.724375,POLITICIAN
62602,Hundreds of parents across the country have ca...,0,0.724375,POLITICIAN


In [20]:
# ─── Extractie span-uri exacte din propozitii ────────────────────────────────

LABEL_TO_GAZETTEER = {
    "POLITICIAN":        ALL_POLITICIANS,
    "POLITICAL_ORG":     POLITICAL_ORGS_GAZETTEER,
    "FINANCIAL_ORG":     FINANCIAL_ORGS_GAZETTEER,
    "ECONOMIC_INDICATOR": ECONOMIC_INDICATORS_GAZETTEER,
    "LEGISLATION":       LEGISLATION_GAZETTEER,
    "CURRENCY":          CURRENCIES_GAZETTEER,
    "TRADE_AGREEMENT":   TRADE_AGREEMENTS_GAZETTEER,
    "MARKET_EVENT":      MARKET_EVENTS_GAZETTEER,
}

def find_entity_spans(text, label):
    """Gaseste toate aparitiile entitatii in text si returneaza span-uri."""
    spans = []
    gazetteer = LABEL_TO_GAZETTEER.get(label, [])

    for entity in gazetteer:
        # Case-insensitive match cu word boundaries
        # cauta exact cuvantul din mijloc, iar escape converteste acel cuvant in string pt a evita cazurile in care contine ,.- etc.

        pattern = r"\b" + re.escape(entity) + r"\b"
        for match in re.finditer(pattern, text, re.IGNORECASE): # ignora diferentele intre literele mari si mici.
            spans.append({
                "text":  match.group(),
                "label": label,
                "start": match.start(),
                "end":   match.end()
            })

    # Policy: pattern special (nu e in gazetteer fix)
    if label == "POLICY":
        policy_patterns = [
            r"\b(quantitative easing|quantitative tightening|rate hike|rate cut|" \
            r"fiscal stimulus|austerity measures|tapering|bailout)\b"
        ]
        for pat in policy_patterns:
            for match in re.finditer(pat, text, re.IGNORECASE):
                spans.append({"text": match.group(), "label": "POLICY",
                               "start": match.start(), "end": match.end()})

    # Elimina duplicate si overlap-uri
    spans.sort(key=lambda s: (s["start"], -(s["end"]-s["start"])))
    unique = []
    last_end = -1
    for s in spans:
        if s["start"] >= last_end:
            unique.append(s)
            last_end = s["end"]
    return unique


def convert_to_ner_format(df):
    """Converteste DataFrame Snorkel in lista de exemple NER cu span-uri."""
    ner_examples = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Extractie entitati"):
        spans = find_entity_spans(row["text"], row["label"])
        if spans:  # pastram doar exemplele cu cel putin o entitate identificata
            ner_examples.append({
                "text":     row["text"],
                "entities": spans,
                "source":   "snorkel_weak_supervision"
            })
    return ner_examples


ner_weak = convert_to_ner_format(df_weak)

# Salvam
with open("dataset/annotated/weak_supervision.jsonl", "w") as f:
    for ex in ner_weak:
        f.write(json.dumps(ex) + "\n")

print(f"Exemple NER cu span-uri extrase: {len(ner_weak)}")
print("\nPrime 2 exemple:")
for ex in ner_weak[:2]:
    print(json.dumps(ex, indent=2))
    print()

Extractie entitati:   0%|          | 0/1261 [00:00<?, ?it/s]

Exemple NER cu span-uri extrase: 1256

Prime 2 exemple:
{
  "text": "President Donald Trump has been pushing hard for tax changes this year.",
  "entities": [
    {
      "text": "Donald Trump",
      "label": "POLITICIAN",
      "start": 10,
      "end": 22
    }
  ],
  "source": "snorkel_weak_supervision"
}

{
  "text": "Days after President Donald Trump deemed Jeff Sessions \"beleaguered\" and threatened to fire him last July, members of the president's inner circle made a desperate case to save the attorney general's job.",
  "entities": [
    {
      "text": "Donald Trump",
      "label": "POLITICIAN",
      "start": 21,
      "end": 33
    }
  ],
  "source": "snorkel_weak_supervision"
}



Pregatire Doccano


In [21]:
# ─── Pregatire fisier import Doccano ─────────────────────────────────────────
# Selectam propozitii reprezentative pentru adnotare manuala.
# Alegem diversitate: propozitii din surse diferite, cu entitati diferite.

# Selectam propozitii reprezentative — unele cu entitati probabile, altele random
high_value = df_sentences[
    df_sentences["text"].str.contains(
        "|".join(["Fed", "GDP", "president", "election", "IMF", "Congress",
                  "inflation", "policy", "sanctions", "treaty"]),
        case=False, regex=True
    )
]["text"].tolist()

random_sample = df_sentences["text"].sample(n=min(200, len(df_sentences)), random_state=42).tolist()
doccano_candidates = list(set(high_value[:800] + random_sample))
random.shuffle(doccano_candidates)
doccano_candidates = doccano_candidates[:1000]  # maxim 1000 pentru adnotare

# Format Doccano: JSONL cu campul 'text'
doccano_path = Path("dataset/annotated/doccano_import.jsonl")
with open(doccano_path, "w") as f:
    for text in doccano_candidates:
        f.write(json.dumps({"text": text}) + "\n")

print(f"Generat fisier pentru import Doccano: {doccano_path}")
print(f"Propozitii de adnotat: {len(doccano_candidates)}")
print(f"\nLabeluri de folosit in Doccano (exact cum sunt definite):")
for label in NER_LABELS:
    print(f"  - {label}")

Generat fisier pentru import Doccano: dataset/annotated/doccano_import.jsonl
Propozitii de adnotat: 996

Labeluri de folosit in Doccano (exact cum sunt definite):
  - POLITICIAN
  - POLITICAL_PARTY
  - POLITICAL_ORG
  - FINANCIAL_ORG
  - ECONOMIC_INDICATOR
  - POLICY
  - LEGISLATION
  - MARKET_EVENT
  - CURRENCY
  - TRADE_AGREEMENT
  - GPE


In [28]:
!pip install label-studio
import subprocess, threading, time
def run():
    subprocess.run(["label-studio", "start", "--port", "8080", "--no-browser"])
threading.Thread(target=run, daemon=True).start()
time.sleep(8)

from pyngrok import ngrok
!pip install pyngrok -q
tunnel = ngrok.connect(8080)
print(tunnel.public_url)

ModuleNotFoundError: No module named 'pyngrok'

Continue without LLM annotated and Gold standard

In [24]:
# ─── Incarcare toate sursele ─────────────────────────────────────────────────

def load_jsonl(path):
    if not Path(path).exists():
        return []
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

weak_examples  = load_jsonl("dataset/annotated/weak_supervision.jsonl")
llm_examples   = load_jsonl("dataset/annotated/llm_annotated.jsonl")
gold_examples  = load_jsonl("dataset/annotated/gold_standard.jsonl")

print("=== Statistici dataset ===\n")
print(f"Weak supervision (Snorkel): {len(weak_examples):>6} exemple")
print(f"LLM annotated (Claude):     {len(llm_examples):>6} exemple")
print(f"Gold standard (Doccano):    {len(gold_examples):>6} exemple")
print(f"{'─'*40}")
print(f"TOTAL:                      {len(weak_examples)+len(llm_examples)+len(gold_examples):>6} exemple")

=== Statistici dataset ===

Weak supervision (Snorkel):   1256 exemple
LLM annotated (Claude):          0 exemple
Gold standard (Doccano):         0 exemple
────────────────────────────────────────
TOTAL:                        1256 exemple


In [25]:
# ─── Validare si curatare ─────────────────────────────────────────────────────
# Aplicam filtre pentru a asigura calitatea dataset-ului.

def validate_example(ex):
    """Returneaza True daca exemplul e valid, False daca trebuie eliminat."""
    text = ex.get("text", "")
    if not text or len(text) < MIN_SENTENCE_LEN:
        return False

    entities = ex.get("entities", [])
    if not entities:  # fara entitati nu e util pentru NER
        return False

    for ent in entities:
        # Verificam ca span-urile sunt valide
        start, end = ent.get("start", 0), ent.get("end", 0)
        if start < 0 or end > len(text) or start >= end:
            return False
        # Verificam ca textul entitati corespunde pozitiei
        if ent.get("text") and ent["text"] != text[start:end]:
            return False
        # Verificam ca labelu e cunoscut
        if ent.get("label") and ent["label"] not in NER_LABELS + ["GPE"]:
            return False

    return True


def deduplicate(examples):
    """Elimina exemplele cu text duplicat."""
    seen, unique = set(), []
    for ex in examples:
        if ex["text"] not in seen:
            seen.add(ex["text"])
            unique.append(ex)
    return unique


# Combinam in ordinea: gold > llm > weak (prioritate la calitate mai mare)
all_examples = gold_examples + llm_examples + weak_examples

# Validare
valid_examples = [ex for ex in all_examples if validate_example(ex)]
print(f"Dupa validare: {len(valid_examples)} / {len(all_examples)} exemple")

# Deduplicare (gold standard are prioritate — apare primul)
unique_examples = deduplicate(valid_examples)
print(f"Dupa deduplicare: {len(unique_examples)} exemple")

# Shuffle final
random.shuffle(unique_examples)

# Distributie labeluri finale
from collections import Counter
label_counts = Counter(
    ent["label"]
    for ex in unique_examples
    for ent in ex["entities"]
)
print("\nDistributie entitati in dataset final:")
for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f"  {label:<25} {count:>6} entitati")

Dupa validare: 1256 / 1256 exemple
Dupa deduplicare: 1256 exemple

Distributie entitati in dataset final:
  POLITICIAN                  1305 entitati
  FINANCIAL_ORG                 29 entitati
  MARKET_EVENT                  20 entitati
  ECONOMIC_INDICATOR            13 entitati


In [26]:
# ─── Train / Dev / Test Split ────────────────────────────────────────────────

from sklearn.model_selection import train_test_split

# Gold standard merge direct in test — nu va fi vazut la antrenare
gold_texts = set(ex["text"] for ex in gold_examples)
test_gold  = [ex for ex in unique_examples if ex["text"] in gold_texts]
rest       = [ex for ex in unique_examples if ex["text"] not in gold_texts]

# ── Split 1: rest → 80% train_full + 20% test_extra ──────────────────────────
train_full, test_extra = train_test_split(
    rest,
    test_size=0.20,
    random_state=42
)

# Test final = gold standard + test extra
test_set = test_gold + test_extra
random.shuffle(test_set)

# ── Split 2: train_full → 80% train + 20% dev ────────────────────────────────
train_set, dev_set = train_test_split(
    train_full,
    test_size=0.20,
    random_state=42
)

# ── Salvare ──────────────────────────────────────────────────────────────────
def save_jsonl(examples, path):
    with open(path, "w") as f:
        for ex in examples:
            f.write(json.dumps(ex) + "\n")
    print(f"Salvat: {path} ({len(examples)} exemple)")

save_jsonl(train_set, "dataset/splits/train.jsonl")
save_jsonl(dev_set,   "dataset/splits/dev.jsonl")
save_jsonl(test_set,  "dataset/splits/test.jsonl")

# ── Sumar final ───────────────────────────────────────────────────────────────
total = len(train_set) + len(dev_set) + len(test_set)
print(f"""
=== SPLIT FINAL ===

  train.jsonl : {len(train_set):>5} exemple  ({len(train_set)/total*100:.1f}%)
  dev.jsonl   : {len(dev_set):>5} exemple  ({len(dev_set)/total*100:.1f}%)
  test.jsonl  : {len(test_set):>5} exemple  ({len(test_set)/total*100:.1f}%)
  ─────────────────────────────
  TOTAL       : {total:>5} exemple

  Din test set: {len(test_gold)} exemple gold standard (Doccano)
""")

Salvat: dataset/splits/train.jsonl (803 exemple)
Salvat: dataset/splits/dev.jsonl (201 exemple)
Salvat: dataset/splits/test.jsonl (252 exemple)

=== SPLIT FINAL ===

  train.jsonl :   803 exemple  (63.9%)
  dev.jsonl   :   201 exemple  (16.0%)
  test.jsonl  :   252 exemple  (20.1%)
  ─────────────────────────────
  TOTAL       :  1256 exemple

  Din test set: 0 exemple gold standard (Doccano)



In [27]:
# ─── Statistici finale si verificare sanity ──────────────────────────────────
# Verificam ca split-urile nu au overlap si distributia labelurilor e echilibrata.

train_texts = set(ex["text"] for ex in train_set)
dev_texts   = set(ex["text"] for ex in dev_set)
test_texts  = set(ex["text"] for ex in test_set)

print("=== Verificare overlap (trebuie sa fie 0) ===")
print(f"  train ∩ dev  : {len(train_texts & dev_texts)}")
print(f"  train ∩ test : {len(train_texts & test_texts)}")
print(f"  dev   ∩ test : {len(dev_texts & test_texts)}")

print("\n=== Distributie entitati per split ===")
for split_name, split_data in [("train", train_set), ("dev", dev_set), ("test", test_set)]:
    counts = Counter(ent["label"] for ex in split_data for ent in ex["entities"])
    total_ents = sum(counts.values())
    print(f"\n  [{split_name}] {total_ents} entitati totale:")
    for label in NER_LABELS + ["GPE"]:
        if label in counts:
            print(f"    {label:<25} {counts[label]:>5}")

print("\n=== Structura directoare finale ===")
for p in sorted(Path("dataset").rglob("*.jsonl")):
    size = sum(1 for _ in open(p))
    print(f"  {p}  ({size} linii)")

print("\nDataset gata. Urmatorul pas: Fine-tuning GLiNER / spaCy.")

=== Verificare overlap (trebuie sa fie 0) ===
  train ∩ dev  : 0
  train ∩ test : 0
  dev   ∩ test : 0

=== Distributie entitati per split ===

  [train] 875 entitati totale:
    POLITICIAN                  831
    FINANCIAL_ORG                20
    ECONOMIC_INDICATOR           10
    MARKET_EVENT                 14

  [dev] 221 entitati totale:
    POLITICIAN                  216
    ECONOMIC_INDICATOR            3
    MARKET_EVENT                  2

  [test] 271 entitati totale:
    POLITICIAN                  258
    FINANCIAL_ORG                 9
    MARKET_EVENT                  4

=== Structura directoare finale ===
  dataset/annotated/doccano_import.jsonl  (996 linii)
  dataset/annotated/weak_supervision.jsonl  (1256 linii)
  dataset/splits/dev.jsonl  (201 linii)
  dataset/splits/test.jsonl  (252 linii)
  dataset/splits/train.jsonl  (803 linii)

Dataset gata. Urmatorul pas: Fine-tuning GLiNER / spaCy.


Pt ca datasetul e neechilibrat va trebui sa mai scoatem din instante si sa mai adaugam cateva din celelalte clase.

In [29]:
# ─── REBALANSARE DATASET ─────────────────────────────────────────────────────
import json, random
from pathlib import Path
from collections import defaultdict

random.seed(42)

# Incarcam weak supervision
with open("dataset/annotated/weak_supervision.jsonl") as f:
    all_examples = [json.loads(l) for l in f if l.strip()]

# Grupam exemplele dupa labelul dominant
by_label = defaultdict(list)
for ex in all_examples:
    if ex["entities"]:
        dominant = ex["entities"][0]["label"]
        by_label[dominant].append(ex)

print("Inainte de rebalansare:")
for label, exs in sorted(by_label.items(), key=lambda x: -len(x[1])):
    print(f"  {label:<25} {len(exs)}")

# Undersampling POLITICIAN la maxim 300
MAX_PER_LABEL = 300
balanced = []
for label, exs in by_label.items():
    if label == "POLITICIAN":
        balanced.extend(random.sample(exs, min(MAX_PER_LABEL, len(exs))))
    else:
        balanced.extend(exs)

random.shuffle(balanced)

# Salvam versiunea balansata
with open("dataset/annotated/weak_supervision_balanced.jsonl", "w") as f:
    for ex in balanced:
        f.write(json.dumps(ex) + "\n")

print(f"\nDupa rebalansare: {len(balanced)} exemple totale")

Inainte de rebalansare:
  POLITICIAN                1206
  FINANCIAL_ORG             23
  MARKET_EVENT              20
  ECONOMIC_INDICATOR        7

Dupa rebalansare: 350 exemple totale


In [30]:
# ─── GENERARE EXEMPLE SINTETICE PENTRU LABELURI ABSENTE ──────────────────────
# Creem manual propozitii reprezentative pentru labelurile slab reprezentate.
# Acestea sunt exemple "silver standard" — calitate buna, nu gold.

import re

def find_spans(text, entities_dict):
    """Gaseste span-urile pentru o lista de (text_entitate, label)."""
    spans = []
    for ent_text, label in entities_dict:
        pattern = r"\b" + re.escape(ent_text) + r"\b"
        for m in re.finditer(pattern, text, re.IGNORECASE):
            spans.append({
                "text": m.group(),
                "label": label,
                "start": m.start(),
                "end": m.end()
            })
    return spans

synthetic_data = []

# ── POLITICAL_PARTY ──────────────────────────────────────────────────────────
party_sentences = [
    ("The Republican Party has opposed the new tax reform bill in Congress.",
     [("Republican Party", "POLITICAL_PARTY")]),
    ("Labour Party members voted against the austerity measures proposed by the government.",
     [("Labour Party", "POLITICAL_PARTY")]),
    ("The Democratic Party introduced a new climate legislation package.",
     [("Democratic Party", "POLITICAL_PARTY")]),
    ("Germany's CDU lost significant support after the coalition talks collapsed.",
     [("CDU", "POLITICAL_PARTY")]),
    ("The Conservative Party announced new fiscal policies ahead of elections.",
     [("Conservative Party", "POLITICAL_PARTY")]),
    ("France's En Marche party secured a parliamentary majority.",
     [("En Marche", "POLITICAL_PARTY")]),
    ("The Green Party pushed for stricter carbon emission targets.",
     [("Green Party", "POLITICAL_PARTY")]),
    ("Members of the Socialist Party demanded higher minimum wages.",
     [("Socialist Party", "POLITICAL_PARTY")]),
]

# ── POLITICAL_ORG ────────────────────────────────────────────────────────────
political_org_sentences = [
    ("NATO allies agreed to increase defense spending to 2% of GDP.",
     [("NATO", "POLITICAL_ORG"), ("GDP", "ECONOMIC_INDICATOR")]),
    ("The European Commission proposed new regulations on digital markets.",
     [("European Commission", "POLITICAL_ORG")]),
    ("The United Nations called for an immediate ceasefire in the conflict region.",
     [("United Nations", "POLITICAL_ORG")]),
    ("G7 leaders agreed on a coordinated response to global inflation.",
     [("G7", "POLITICAL_ORG"), ("inflation", "ECONOMIC_INDICATOR")]),
    ("The European Parliament voted to approve the new budget framework.",
     [("European Parliament", "POLITICAL_ORG")]),
    ("The G20 summit focused on global debt restructuring mechanisms.",
     [("G20", "POLITICAL_ORG")]),
    ("OECD revised its global growth forecast downward for the third quarter.",
     [("OECD", "POLITICAL_ORG")]),
    ("The WTO ruled against the trade tariffs imposed by the United States.",
     [("WTO", "POLITICAL_ORG"), ("United States", "GPE")]),
]

# ── LEGISLATION ──────────────────────────────────────────────────────────────
legislation_sentences = [
    ("The Dodd-Frank Act imposed stricter regulations on Wall Street banks.",
     [("Dodd-Frank Act", "LEGISLATION")]),
    ("Congress passed the Inflation Reduction Act to address rising consumer prices.",
     [("Inflation Reduction Act", "LEGISLATION")]),
    ("The CARES Act provided $2 trillion in economic relief during the pandemic.",
     [("CARES Act", "LEGISLATION")]),
    ("Basel III requirements forced banks to hold higher capital reserves.",
     [("Basel III", "LEGISLATION")]),
    ("The Sarbanes-Oxley Act strengthened corporate governance and auditing standards.",
     [("Sarbanes-Oxley Act", "LEGISLATION")]),
    ("The Glass-Steagall Act separated commercial and investment banking activities.",
     [("Glass-Steagall Act", "LEGISLATION")]),
    ("New MiFID regulations changed how financial instruments are traded in Europe.",
     [("MiFID", "LEGISLATION")]),
    ("The American Rescue Plan allocated funds for vaccine distribution and economic recovery.",
     [("American Rescue Plan", "LEGISLATION")]),
]

# ── CURRENCY ─────────────────────────────────────────────────────────────────
currency_sentences = [
    ("The EUR weakened against the USD following the ECB's rate decision.",
     [("EUR", "CURRENCY"), ("USD", "CURRENCY"), ("ECB", "FINANCIAL_ORG")]),
    ("Bitcoin surged past $60,000 amid institutional buying pressure.",
     [("Bitcoin", "CURRENCY")]),
    ("The Chinese yuan depreciated as trade tensions with the US escalated.",
     [("yuan", "CURRENCY")]),
    ("Sterling fell to its lowest level against the dollar in two decades.",
     [("Sterling", "CURRENCY"), ("dollar", "CURRENCY")]),
    ("The yen hit a 30-year low against the dollar as the Bank of Japan maintained low rates.",
     [("yen", "CURRENCY"), ("dollar", "CURRENCY")]),
    ("Ethereum's market capitalization exceeded $500 billion for the first time.",
     [("Ethereum", "CURRENCY")]),
    ("The Swiss franc remained a safe-haven currency during the market turmoil.",
     [("Swiss franc", "CURRENCY")]),
    ("Oil prices are denominated in USD, affecting countries with other currencies.",
     [("USD", "CURRENCY")]),
]

# ── TRADE_AGREEMENT ──────────────────────────────────────────────────────────
trade_sentences = [
    ("NAFTA was replaced by the USMCA in 2020, updating trade rules for North America.",
     [("NAFTA", "TRADE_AGREEMENT"), ("USMCA", "TRADE_AGREEMENT")]),
    ("The Paris Agreement committed nations to limiting global warming to 1.5 degrees.",
     [("Paris Agreement", "TRADE_AGREEMENT")]),
    ("Negotiations for the TTIP stalled over regulatory differences between the EU and US.",
     [("TTIP", "TRADE_AGREEMENT")]),
    ("The CPTPP created a major trade bloc across Pacific nations.",
     [("CPTPP", "TRADE_AGREEMENT")]),
    ("The Maastricht Treaty established the framework for European monetary union.",
     [("Maastricht Treaty", "TRADE_AGREEMENT")]),
    ("RCEP became the world's largest trade agreement by GDP covered.",
     [("RCEP", "TRADE_AGREEMENT"), ("GDP", "ECONOMIC_INDICATOR")]),
    ("The WTO Agreement on Agriculture remains a contentious point in trade talks.",
     [("WTO Agreement on Agriculture", "TRADE_AGREEMENT")]),
]

# ── GPE (Geopolitical Entities) ──────────────────────────────────────────────
gpe_sentences = [
    ("The United States imposed new tariffs on Chinese goods.",
     [("United States", "GPE")]),
    ("Germany recorded a budget surplus for the fifth consecutive year.",
     [("Germany", "GPE")]),
    ("The Eurozone economy contracted by 0.3% in the second quarter.",
     [("Eurozone", "ECONOMIC_INDICATOR")]),
    ("China's GDP growth slowed to 4.5% amid property sector concerns.",
     [("China", "GPE"), ("GDP", "ECONOMIC_INDICATOR")]),
    ("The European Union launched an antitrust investigation into tech giants.",
     [("European Union", "GPE")]),
    ("Japan's central bank kept interest rates near zero despite rising inflation.",
     [("Japan", "GPE"), ("inflation", "ECONOMIC_INDICATOR")]),
]

# ── ECONOMIC_INDICATOR (suplimentar) ─────────────────────────────────────────
econ_sentences = [
    ("US GDP grew by 2.1% in the third quarter, beating analyst expectations.",
     [("GDP", "ECONOMIC_INDICATOR")]),
    ("The CPI rose 0.4% month-over-month, signaling persistent inflationary pressure.",
     [("CPI", "ECONOMIC_INDICATOR"), ("inflationary", "ECONOMIC_INDICATOR")]),
    ("Unemployment rate fell to 3.7%, the lowest in 50 years.",
     [("Unemployment rate", "ECONOMIC_INDICATOR")]),
    ("The Federal Reserve raised the federal funds rate by 25 basis points.",
     [("federal funds rate", "ECONOMIC_INDICATOR"), ("Federal Reserve", "FINANCIAL_ORG")]),
    ("Bond yields rose sharply after stronger-than-expected jobs data.",
     [("Bond yields", "ECONOMIC_INDICATOR")]),
    ("The trade deficit widened to $80 billion as imports outpaced exports.",
     [("trade deficit", "ECONOMIC_INDICATOR")]),
    ("Core PCE inflation, the Fed's preferred gauge, came in at 2.8%.",
     [("PCE inflation", "ECONOMIC_INDICATOR"), ("Fed", "FINANCIAL_ORG")]),
    ("The PMI index dropped below 50, indicating contraction in manufacturing.",
     [("PMI", "ECONOMIC_INDICATOR")]),
]

# ── MARKET_EVENT (suplimentar) ────────────────────────────────────────────────
market_sentences = [
    ("The 2008 financial crisis wiped out trillions of dollars in global wealth.",
     [("2008 financial crisis", "MARKET_EVENT")]),
    ("The dot-com bubble burst in 2000 led to massive layoffs in the tech sector.",
     [("dot-com bubble", "MARKET_EVENT")]),
    ("Brexit caused significant volatility in currency and equity markets.",
     [("Brexit", "MARKET_EVENT")]),
    ("The Great Recession forced governments to implement unprecedented stimulus measures.",
     [("Great Recession", "MARKET_EVENT")]),
    ("Black Monday in 1987 saw stock markets lose over 20% in a single day.",
     [("Black Monday", "MARKET_EVENT")]),
]

# ── POLICY (suplimentar) ─────────────────────────────────────────────────────
policy_sentences = [
    ("The Fed announced a new round of quantitative easing to support the economy.",
     [("quantitative easing", "POLICY"), ("Fed", "FINANCIAL_ORG")]),
    ("Austerity measures imposed by the government led to widespread protests.",
     [("Austerity measures", "POLICY")]),
    ("The central bank began tapering its bond purchases ahead of rate hikes.",
     [("tapering", "POLICY")]),
    ("Fiscal stimulus of $1.9 trillion was approved to boost post-pandemic recovery.",
     [("Fiscal stimulus", "POLICY")]),
    ("Quantitative tightening began as inflation remained above the 2% target.",
     [("Quantitative tightening", "POLICY"), ("inflation", "ECONOMIC_INDICATOR")]),
    ("The government announced a bailout package for the struggling banking sector.",
     [("bailout", "POLICY")]),
]

# ── Combinam toate propozitiile sintetice ────────────────────────────────────
all_synthetic_groups = (
    party_sentences + political_org_sentences + legislation_sentences +
    currency_sentences + trade_sentences + gpe_sentences +
    econ_sentences + market_sentences + policy_sentences
)

for text, entities_dict in all_synthetic_groups:
    spans = find_spans(text, entities_dict)
    if spans:
        synthetic_data.append({
            "text": text,
            "entities": spans,
            "source": "synthetic_template"
        })

with open("dataset/annotated/synthetic.jsonl", "w") as f:
    for ex in synthetic_data:
        f.write(json.dumps(ex) + "\n")

print(f"Generate {len(synthetic_data)} exemple sintetice")

from collections import Counter
label_counts = Counter(
    ent["label"]
    for ex in synthetic_data
    for ent in ex["entities"]
)
print("\nDistributie labeluri in exemple sintetice:")
for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f"  {label:<25} {count}")

Generate 64 exemple sintetice

Distributie labeluri in exemple sintetice:
  ECONOMIC_INDICATOR        16
  CURRENCY                  11
  POLITICAL_PARTY           8
  POLITICAL_ORG             8
  LEGISLATION               8
  TRADE_AGREEMENT           8
  GPE                       6
  POLICY                    6
  MARKET_EVENT              5
  FINANCIAL_ORG             4


Reluam pasul 9 ca sa incarcam si aceste surse

In [36]:
# ─── Incarcare toate sursele ─────────────────────────────────────────────────

def load_jsonl(path):
    if not Path(path).exists():
        return []
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

weak_examples  = load_jsonl("dataset/annotated/weak_supervision_balanced.jsonl")
synthetic_examples = load_jsonl("dataset/annotated/synthetic.jsonl")
augmented_examples = load_jsonl("dataset/annotated/synthetic_augmented.jsonl")
llm_examples   = load_jsonl("dataset/annotated/llm_annotated.jsonl")
gold_examples  = load_jsonl("dataset/annotated/gold_standard.jsonl")

print("=== Statistici dataset ===\n")
print(f"Weak supervision (Snorkel): {len(weak_examples):>6} exemple")
print(f"LLM annotated (Claude):     {len(llm_examples):>6} exemple")
print(f"Gold standard (Doccano):    {len(gold_examples):>6} exemple")
print(f"Syntetic Examples:          {len(synthetic_examples):>6} exemple")
print(f"Augmented Examples:         {len(augmented_examples):>6} exemple")
print(f"{'─'*40}")
print(f"TOTAL:                      {len(weak_examples)+len(llm_examples)+len(gold_examples)+len(synthetic_examples)+len(augmented_examples):>6} exemple")

=== Statistici dataset ===

Weak supervision (Snorkel):    350 exemple
LLM annotated (Claude):          0 exemple
Gold standard (Doccano):         0 exemple
Syntetic Examples:              64 exemple
Augmented Examples:            385 exemple
────────────────────────────────────────
TOTAL:                         799 exemple


In [37]:
# ─── Validare si curatare ─────────────────────────────────────────────────────
# Aplicam filtre pentru a asigura calitatea dataset-ului.

def validate_example(ex):
    """Returneaza True daca exemplul e valid, False daca trebuie eliminat."""
    text = ex.get("text", "")
    if not text or len(text) < MIN_SENTENCE_LEN:    # eliminam daca are o lungime mai mica
        return False

    entities = ex.get("entities", [])       # extrage doar atributul entities
    if not entities:  # fara entitati nu e util pentru NER
        return False

    for ent in entities:
        # Verificam ca span-urile sunt valide
        start, end = ent.get("start", 0), ent.get("end", 0)
        if start < 0 or end > len(text) or start >= end:
            return False
        # Verificam ca textul entitati corespunde pozitiei
        if ent.get("text") and ent["text"] != text[start:end]:
            return False
        # Verificam ca labelul e cunoscut
        if ent.get("label") and ent["label"] not in NER_LABELS + ["GPE"]:
            return False

    return True


def deduplicate(examples):
    """Elimina exemplele cu text duplicat."""
    seen, unique = set(), []
    for ex in examples:
        if ex["text"] not in seen:
            seen.add(ex["text"])
            unique.append(ex)
    return unique


# Combinam in ordinea: gold > llm > weak (prioritate la calitate mai mare)
all_examples = gold_examples + llm_examples + weak_examples + synthetic_examples + augmented_examples

# Validare
valid_examples = [ex for ex in all_examples if validate_example(ex)]
print(f"Dupa validare: {len(valid_examples)} / {len(all_examples)} exemple")

# Deduplicare (gold standard are prioritate — apare primul)
unique_examples = deduplicate(valid_examples)
print(f"Dupa deduplicare: {len(unique_examples)} exemple")

# Shuffle final
random.shuffle(unique_examples)

# Distributie labeluri finale
from collections import Counter
label_counts = Counter(
    ent["label"]
    for ex in unique_examples
    for ent in ex["entities"]
)
print("\nDistributie entitati in dataset final:")
for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f"  {label:<25} {count:>6} entitati")

Dupa validare: 799 / 799 exemple
Dupa deduplicare: 799 exemple

Distributie entitati in dataset final:
  POLITICIAN                   320 entitati
  POLITICAL_ORG                 87 entitati
  GPE                           82 entitati
  CURRENCY                      77 entitati
  POLITICAL_PARTY               65 entitati
  LEGISLATION                   65 entitati
  TRADE_AGREEMENT               63 entitati
  POLICY                        61 entitati
  FINANCIAL_ORG                 48 entitati
  ECONOMIC_INDICATOR            32 entitati
  MARKET_EVENT                  25 entitati


Pt ca distributia e tot mica, s-a utilizat LLM-ul Claude pt a genera inca 385 de exemple (cate 50 in medie din fiecare clasa).

Facem din nou split la aceste date

In [38]:
# ─── Train / Dev / Test Split ────────────────────────────────────────────────

from sklearn.model_selection import train_test_split

# Gold standard merge direct in test — nu va fi vazut la antrenare
gold_texts = set(ex["text"] for ex in gold_examples)
test_gold  = [ex for ex in unique_examples if ex["text"] in gold_texts]
rest       = [ex for ex in unique_examples if ex["text"] not in gold_texts]

# ── Split 1: rest → 80% train_full + 20% test_extra ──────────────────────────
train_full, test_extra = train_test_split(
    rest,
    test_size=0.20,
    random_state=42
)

# Test final = gold standard + test extra
test_set = test_gold + test_extra
random.shuffle(test_set)

# ── Split 2: train_full → 80% train + 20% dev ────────────────────────────────
train_set, dev_set = train_test_split(
    train_full,
    test_size=0.20,
    random_state=42
)

# ── Salvare ──────────────────────────────────────────────────────────────────
def save_jsonl(examples, path):
    with open(path, "w") as f:
        for ex in examples:
            f.write(json.dumps(ex) + "\n")
    print(f"Salvat: {path} ({len(examples)} exemple)")

save_jsonl(train_set, "dataset/splits/train.jsonl")
save_jsonl(dev_set,   "dataset/splits/dev.jsonl")
save_jsonl(test_set,  "dataset/splits/test.jsonl")

# ── Sumar final ───────────────────────────────────────────────────────────────
total = len(train_set) + len(dev_set) + len(test_set)
print(f"""
=== SPLIT FINAL ===

  train.jsonl : {len(train_set):>5} exemple  ({len(train_set)/total*100:.1f}%)
  dev.jsonl   : {len(dev_set):>5} exemple  ({len(dev_set)/total*100:.1f}%)
  test.jsonl  : {len(test_set):>5} exemple  ({len(test_set)/total*100:.1f}%)
  ─────────────────────────────
  TOTAL       : {total:>5} exemple

  Din test set: {len(test_gold)} exemple gold standard (Doccano)
""")

Salvat: dataset/splits/train.jsonl (511 exemple)
Salvat: dataset/splits/dev.jsonl (128 exemple)
Salvat: dataset/splits/test.jsonl (160 exemple)

=== SPLIT FINAL ===

  train.jsonl :   511 exemple  (64.0%)
  dev.jsonl   :   128 exemple  (16.0%)
  test.jsonl  :   160 exemple  (20.0%)
  ─────────────────────────────
  TOTAL       :   799 exemple

  Din test set: 0 exemple gold standard (Doccano)



In [39]:
# ─── Statistici finale si verificare sanity ──────────────────────────────────
# Verificam ca split-urile nu au overlap si distributia labelurilor e echilibrata.

train_texts = set(ex["text"] for ex in train_set)
dev_texts   = set(ex["text"] for ex in dev_set)
test_texts  = set(ex["text"] for ex in test_set)

print("=== Verificare overlap (trebuie sa fie 0) ===")
print(f"  train ∩ dev  : {len(train_texts & dev_texts)}")
print(f"  train ∩ test : {len(train_texts & test_texts)}")
print(f"  dev   ∩ test : {len(dev_texts & test_texts)}")

print("\n=== Distributie entitati per split ===")
for split_name, split_data in [("train", train_set), ("dev", dev_set), ("test", test_set)]:
    counts = Counter(ent["label"] for ex in split_data for ent in ex["entities"])
    total_ents = sum(counts.values())
    print(f"\n  [{split_name}] {total_ents} entitati totale:")
    for label in NER_LABELS + ["GPE"]:
        if label in counts:
            print(f"    {label:<25} {counts[label]:>5}")

print("\n=== Structura directoare finale ===")
for p in sorted(Path("dataset").rglob("*.jsonl")):
    size = sum(1 for _ in open(p))
    print(f"  {p}  ({size} linii)")

print("\nDataset gata. Urmatorul pas: Fine-tuning GLiNER / spaCy.")

=== Verificare overlap (trebuie sa fie 0) ===
  train ∩ dev  : 0
  train ∩ test : 0
  dev   ∩ test : 0

=== Distributie entitati per split ===

  [train] 593 entitati totale:
    POLITICIAN                  213
    POLITICAL_PARTY              39
    POLITICAL_ORG                56
    FINANCIAL_ORG                29
    ECONOMIC_INDICATOR           24
    POLICY                       35
    LEGISLATION                  43
    MARKET_EVENT                 16
    CURRENCY                     46
    TRADE_AGREEMENT              41
    GPE                          51
    GPE                          51

  [dev] 143 entitati totale:
    POLITICIAN                   52
    POLITICAL_PARTY              13
    POLITICAL_ORG                10
    FINANCIAL_ORG                10
    ECONOMIC_INDICATOR            6
    POLICY                       12
    LEGISLATION                   4
    MARKET_EVENT                  4
    CURRENCY                     15
    TRADE_AGREEMENT               8
   

Salvam si statisticile despre aceste fisiere

In [40]:
import json
from collections import Counter
from pathlib import Path

def load_jsonl(path):
    if not Path(path).exists(): return []
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

splits = {
    "train": load_jsonl("dataset/splits/train.jsonl"),
    "dev":   load_jsonl("dataset/splits/dev.jsonl"),
    "test":  load_jsonl("dataset/splits/test.jsonl"),
}

stats = {}
for split_name, data in splits.items():
    label_counts = Counter(
        ent["label"]
        for ex in data
        for ent in ex["entities"]
    )
    source_counts = Counter(ex.get("source", "unknown") for ex in data)
    stats[split_name] = {
        "n_examples":       len(data),
        "n_entities":       sum(len(ex["entities"]) for ex in data),
        "label_distribution": dict(label_counts),
        "source_distribution": dict(source_counts),
    }

stats["total"] = {
    "n_examples": sum(s["n_examples"] for s in stats.values()),
    "labels": [
        "POLITICIAN", "POLITICAL_PARTY", "POLITICAL_ORG",
        "FINANCIAL_ORG", "ECONOMIC_INDICATOR", "POLICY",
        "LEGISLATION", "MARKET_EVENT", "CURRENCY",
        "TRADE_AGREEMENT", "GPE"
    ]
}

with open("dataset/dataset_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print(json.dumps(stats, indent=2))

{
  "train": {
    "n_examples": 511,
    "n_entities": 593,
    "label_distribution": {
      "POLITICIAN": 213,
      "POLICY": 35,
      "GPE": 51,
      "POLITICAL_ORG": 56,
      "TRADE_AGREEMENT": 41,
      "FINANCIAL_ORG": 29,
      "POLITICAL_PARTY": 39,
      "MARKET_EVENT": 16,
      "ECONOMIC_INDICATOR": 24,
      "CURRENCY": 46,
      "LEGISLATION": 43
    },
    "source_distribution": {
      "snorkel_weak_supervision": 232,
      "synthetic_augmented": 235,
      "synthetic_template": 44
    }
  },
  "dev": {
    "n_examples": 128,
    "n_entities": 143,
    "label_distribution": {
      "LEGISLATION": 4,
      "POLITICIAN": 52,
      "CURRENCY": 15,
      "FINANCIAL_ORG": 10,
      "TRADE_AGREEMENT": 8,
      "POLICY": 12,
      "POLITICAL_PARTY": 13,
      "ECONOMIC_INDICATOR": 6,
      "POLITICAL_ORG": 10,
      "MARKET_EVENT": 4,
      "GPE": 9
    },
    "source_distribution": {
      "synthetic_augmented": 59,
      "snorkel_weak_supervision": 60,
      "synthetic_t

Pt train set:
```txt
LABEL                train  dev  test   TOTAL   STATUS
─────────────────────────────────────────────────────────
POLITICIAN             213   52    55     320   ✅ solid
POLITICAL_ORG           56   10    21      87   ✅ ok
GPE                     51    9    22      82   ✅ ok
CURRENCY                46   15    16      77   ✅ ok
POLITICAL_PARTY         39   13    13      65   ✅ ok
LEGISLATION             43    4    18      65   ✅ ok
TRADE_AGREEMENT         41    8    14      63   ✅ ok
POLICY                  35   12    14      61   ✅ ok
FINANCIAL_ORG           29   10     9      48   ⚠️ limitat
ECONOMIC_INDICATOR      24    6     2      32   ⚠️ limitat
MARKET_EVENT            16    4     5      25   ⚠️ limitat
─────────────────────────────────────────────────────────
TOTAL                  593  143   189     925
```